In [37]:
import numpy as np
import pandas as pd


Recomienda top-k combinaciones de universidad y grado ponderando probabilidad de admisión y prestigio.
    
α: peso de la probabilidad de admisión (ej. 0.7)

β: peso del prestigio (qs_score normalizado) (ej. 0.3)

In [ ]:
recom_score = α * prob + β * (qs_score_normalizado)

def recomendar_top_k_ponderado(perfil_estudiante, df_programas, modelo_rf,
                               le_bach, le_degree, le_university, k=10,
                               alpha=0.7, beta=0.3):
    """
    Recomienda top-k combinaciones de universidad y grado ponderando probabilidad de admisión y prestigio.
    
    α: peso de la probabilidad de admisión (ej. 0.7)
    β: peso del prestigio (qs_score normalizado) (ej. 0.3)
    """
    # 1. Obtener combinaciones únicas
    combinaciones = df_programas[["university", "degree", "qs_score"]].drop_duplicates().copy()

    # 2. Añadir perfil del estudiante
    for key in ["gpa", "toefl", "sat", "profile", "peak", "sat_required"]:
        combinaciones[key] = perfil_estudiante[key]
    combinaciones["bach"] = le_bach.transform([perfil_estudiante["bach"]])[0]

    # 3. Codificar universidad y grado
    combinaciones["university"] = le_university.transform(combinaciones["university"])
    combinaciones["degree"] = le_degree.transform(combinaciones["degree"])

    # 4. Predecir probabilidad de admisión
    features = ['bach', 'gpa', 'toefl', 'sat', 'profile', 'peak',
                'university', 'degree', 'sat_required', 'qs_score']
    combinaciones["prob"] = modelo_rf.predict_proba(combinaciones[features])[:, 1]

    # 5. Normalizar qs_score
    qs_min = combinaciones["qs_score"].min()
    qs_max = combinaciones["qs_score"].max()
    combinaciones["qs_score_norm"] = (combinaciones["qs_score"] - qs_min) / (qs_max - qs_min)

    # 6. Calcular recom_score
    combinaciones["recom_score"] = alpha * combinaciones["prob"] + beta * combinaciones["qs_score_norm"]

    # 7. Decodificar labels
    combinaciones["university"] = le_university.inverse_transform(combinaciones["university"])
    combinaciones["degree"] = le_degree.inverse_transform(combinaciones["degree"])

    # 8. Top-K
    return combinaciones[["degree", "university", "qs_score", "prob", "qs_score_norm", "recom_score"]]\
        .sort_values(by="recom_score", ascending=False).head(k)



In [ ]:
{"gpa": 3.7, "sat": 1350, "toefl": 100, "profile": 12} #...}

probs = model.predict_proba(df_qs_simulado)[:, 1] #modelo probabilistico

relevance_score = (alpha * probabilidad) + (beta * (qs_score_normalizado))

df_resultados['relevance_score'] = 0.7 * df_resultados['admission_prob'] + 0.3 * (df_resultados['qs_score'] / 100)
top_k = df_resultados.sort_values(by="relevance_score", ascending=False).head(K)


